# 7CS537 CW1, Notebook 01: Data Exploration and Cleaning (CICIDS2017)

**Purpose:** load the eight CICIDS2017 MachineLearningCSV files, examine data quality, clean the data and save a cleaned dataset for model training (Notebook 02).

**Case study framing:** CICIDS2017 stands in for TeleComNet Ltd's network flow data. Each row is one network flow produced by CICFlowMeter, labelled as BENIGN or as a specific attack.

**Outputs saved to `outputs/`:** summary tables (CSV) and figures (PNG) for use as testing evidence in the report and presentation.

**Key design decisions (each one is logged with before and after counts):**
1. All eight files are used, because each day contains different attack types. Using one file would leave most attack classes out.
2. Files are converted to `float32` one at a time, so only one full copy of the data is held in memory.
3. Rows with missing or infinite values are removed, then exact duplicates, then flows whose identical features carry conflicting labels.
4. Duplicated and constant columns are removed.
5. `Destination Port` is kept as a metadata column but excluded from the model features, so Notebook 02 can test the effect of including it (an ablation).

Run the cells in order.

## 1. Setup and configuration

In [ ]:
import os
import glob
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:          # allows the code to run as a plain script too
    display = print

# Folder containing the 8 CSV files from MachineLearningCSV.zip (MachineLearningCVE folder)
DATA_DIR = "data/MachineLearningCVE"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)

# Destination Port is excluded from the model features by default because it can act as a
# shortcut (in the CICIDS2017 lab, specific attacks were sent to specific ports).
# It is still saved as metadata so Notebook 02 can run a with/without comparison.
EXCLUDE_DESTINATION_PORT = True

# Identical feature vectors with different labels cannot be learned correctly by any model.
DROP_CONFLICTING_LABELS = True

LABEL_COL = "Label"
PORT_COL = "Destination Port"
RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
print("pandas", pd.__version__, "| numpy", np.__version__)

def mem_mb(frame):
    return frame.memory_usage(deep=True).sum() / 1024**2

## 2. Load all eight CSV files
Each file is cleaned of whitespace in column names, then its features are converted to numbers (`float32`) straight away. Missing and infinite values are counted per file **before** any change, so the report can state exactly how many there were.

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
print(f"Found {len(files)} files")
assert len(files) > 0, "No CSV files found. Check DATA_DIR."

t0 = time.time()
frames, file_summary, nan_counts, inf_counts = [], [], [], []
for f in files:
    # latin-1 avoids decode errors on the dash character in the Web Attack labels
    part = pd.read_csv(f, encoding="latin-1", low_memory=False)
    part.columns = part.columns.str.strip()                      # column names have leading spaces
    part = part.loc[:, ~part.columns.str.match(r"^Unnamed")]      # guard against stray index columns
    feats = [c for c in part.columns if c != LABEL_COL]

    # Coerce to numbers; anything unparseable becomes NaN and is counted below
    num = part[feats].apply(pd.to_numeric, errors="coerce").astype(np.float64)
    nan_counts.append(num.isna().sum().rename(os.path.basename(f)))
    inf_counts.append(np.isinf(num).sum().rename(os.path.basename(f)))
    num = num.replace([np.inf, -np.inf], np.nan).astype(np.float32)

    num[LABEL_COL] = part[LABEL_COL].astype(str).str.strip().values
    num["source_file"] = os.path.basename(f)
    frames.append(num)
    file_summary.append({"file": os.path.basename(f), "rows": len(part), "columns": len(feats) + 1})
    del part

df = pd.concat(frames, ignore_index=True)
del frames
df["source_file"] = df["source_file"].astype("category")

file_summary = pd.DataFrame(file_summary)
file_summary.to_csv(f"{OUT_DIR}/01_files_loaded.csv", index=False)
display(file_summary)
print(f"Combined shape: {df.shape} | memory: {mem_mb(df):,.0f} MB | load time: {time.time()-t0:.0f}s")

## 3. Clean the label column
The Web Attack labels contain a non-standard dash character, so they are normalised to readable names.

In [ ]:
def normalise_label(lbl):
    if lbl.startswith("Web Attack"):
        if "Brute" in lbl: return "Web Attack - Brute Force"
        if "XSS" in lbl:   return "Web Attack - XSS"
        if "Sql" in lbl:   return "Web Attack - Sql Injection"
    return lbl

# Map the handful of unique labels rather than every row (much faster)
label_map = {l: normalise_label(l) for l in df[LABEL_COL].unique()}
df[LABEL_COL] = df[LABEL_COL].map(label_map)
print(sorted(df[LABEL_COL].unique()))

## 4. Raw class distribution (before cleaning)

In [ ]:
raw_counts = df[LABEL_COL].value_counts().rename("raw_count").to_frame()
raw_counts["raw_pct"] = (raw_counts["raw_count"] / len(df) * 100).round(4)
display(raw_counts)

# Which attacks appear in which file (day of capture)
label_by_file = pd.crosstab(df[LABEL_COL], df["source_file"])
label_by_file.to_csv(f"{OUT_DIR}/01_label_by_file.csv")
display(label_by_file)

## 5. Data quality checks
These checks only measure problems. Nothing is removed until Section 6.

In [ ]:
measured_cols = [c for c in df.columns if c not in (LABEL_COL, "source_file")]

# 5a. Missing and infinite values (counted at load time, before conversion)
quality = pd.DataFrame({
    "nan_count": pd.concat(nan_counts, axis=1).sum(axis=1),
    "inf_count": pd.concat(inf_counts, axis=1).sum(axis=1),
    "n_unique": df[measured_cols].nunique(dropna=True),
})
quality = quality[(quality.nan_count > 0) | (quality.inf_count > 0) | (quality.n_unique <= 1)]
quality.to_csv(f"{OUT_DIR}/01_quality_issues.csv")
print("Columns with missing, infinite or constant values:")
display(quality)

# 5b. Negative values. These are kept: in CICFlowMeter output, -1 in the
# Init_Win_bytes columns means the value was not observed, not that the row is corrupt.
negative_cols = (df[measured_cols] < 0).sum()
negative_cols = negative_cols[negative_cols > 0]
negative_cols.rename("negative_count").to_csv(f"{OUT_DIR}/01_negative_values.csv")
print("Columns containing negative values:")
display(negative_cols)

# 5c. Exact duplicate rows (same measurements and same label)
dup_rows = df.duplicated(subset=measured_cols + [LABEL_COL]).sum()
print(f"Exact duplicate rows: {dup_rows:,} ({dup_rows/len(df)*100:.2f}%)")

## 6. Cleaning steps
Each step logs the row or column counts so the report can state exactly what was removed and why.

In [ ]:
log = []
def record(step, reason, before, after, unit="rows"):
    log.append({"step": step, "reason": reason, "before": before, "after": after,
                "removed": before - after, "unit": unit})
    print(f"{step}: {before:,} -> {after:,} {unit} (removed {before - after:,})")

# 6a. Rows with NaN or infinite values (infinities were already converted to NaN)
before = len(df)
df = df.dropna(subset=measured_cols)
record("Drop rows with NaN or infinite values",
       "Undefined rate features (e.g. Flow Bytes/s when duration is 0); too few rows to justify imputation",
       before, len(df))

# 6b. Exact duplicate rows. Prevents the same flow appearing in both train and test,
# which would inflate test scores.
before = len(df)
df = df.drop_duplicates(subset=measured_cols + [LABEL_COL])
record("Drop exact duplicate rows", "Avoid train/test leakage and over-weighting repeated flows",
       before, len(df))

# 6c. Conflicting labels: the same measurements labelled differently
conflict_mask = df.duplicated(subset=measured_cols, keep=False)
conflicts = df.loc[conflict_mask, LABEL_COL].value_counts()
conflicts.rename("rows_in_conflict").to_csv(f"{OUT_DIR}/01_conflicting_labels.csv")
print("Rows whose identical measurements carry more than one label:")
display(conflicts)
if DROP_CONFLICTING_LABELS:
    before = len(df)
    df = df.loc[~conflict_mask]
    record("Drop rows with conflicting labels",
           "Ground truth is ambiguous; no model can classify these correctly", before, len(df))
del conflict_mask

# 6d. Duplicated and constant columns (checked on the cleaned rows)
dup_cols, seen = [], {}
for c in measured_cols:
    key = int(pd.util.hash_pandas_object(df[c], index=False).sum())
    if key in seen and df[c].equals(df[seen[key]]):
        dup_cols.append((c, seen[key]))
    else:
        seen[key] = c
print("Duplicated columns (column, copy of):", dup_cols)
constant_cols = [c for c in measured_cols if df[c].nunique() <= 1]
print("Constant columns:", constant_cols)

drop_cols = [c for c, _ in dup_cols] + [c for c in constant_cols if c not in dict(dup_cols)]
before_cols = len(measured_cols)
df = df.drop(columns=drop_cols)
measured_cols = [c for c in measured_cols if c not in drop_cols]
record("Drop duplicated and constant columns", "Carry no extra information", before_cols,
       len(measured_cols), unit="columns")

# 6e. Destination Port: kept as metadata, excluded from features
feature_cols = list(measured_cols)
if EXCLUDE_DESTINATION_PORT and PORT_COL in feature_cols:
    feature_cols.remove(PORT_COL)
    record("Exclude Destination Port from features",
           "Shortcut risk: lab attacks targeted fixed ports; kept as metadata for an ablation",
           len(measured_cols), len(feature_cols), unit="columns")

pd.DataFrame(log).to_csv(f"{OUT_DIR}/01_cleaning_log.csv", index=False)
df = df.reset_index(drop=True)
print(f"Final shape: {df.shape} | memory: {mem_mb(df):,.0f} MB | model features: {len(feature_cols)}")

## 7. Target definitions
Two targets are created so Notebook 02 can choose between them:
- **`is_attack`** (binary): the main detection task, BENIGN versus any attack.
- **`attack_family`**: groups the 15 labels into families, used to break down detection performance by attack type (for example, does the model miss rare web attacks?).

In [ ]:
family_map = {
    "BENIGN": "Benign",
    "DoS Hulk": "DoS", "DoS GoldenEye": "DoS", "DoS slowloris": "DoS", "DoS Slowhttptest": "DoS",
    "DDoS": "DDoS",
    "PortScan": "PortScan",
    "FTP-Patator": "Brute Force", "SSH-Patator": "Brute Force",
    "Web Attack - Brute Force": "Web Attack", "Web Attack - XSS": "Web Attack",
    "Web Attack - Sql Injection": "Web Attack",
    "Bot": "Botnet",
    "Infiltration": "Infiltration",
    "Heartbleed": "Heartbleed",
}
unmapped = set(df[LABEL_COL]) - set(family_map)
assert not unmapped, f"Unmapped labels: {unmapped}"

df["attack_family"] = df[LABEL_COL].map(family_map).astype("category")
df["is_attack"] = (df[LABEL_COL] != "BENIGN").astype(np.int8)
df[LABEL_COL] = df[LABEL_COL].astype("category")

clean_counts = df[LABEL_COL].value_counts().rename("clean_count").to_frame()
dist = raw_counts.join(clean_counts, how="left").fillna(0)
dist["removed"] = dist["raw_count"] - dist["clean_count"]
dist["clean_pct"] = (dist["clean_count"] / dist["clean_count"].sum() * 100).round(4)
dist["family"] = dist.index.map(family_map)
dist.to_csv(f"{OUT_DIR}/01_class_distribution.csv")
display(dist)

print(df["is_attack"].value_counts(normalize=True).rename({0: "Benign", 1: "Attack"}))

## 8. Visualising the class imbalance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
d = dist.sort_values("clean_count")
ax.barh(d.index, d["clean_count"])
ax.set_xscale("log")
ax.set_xlabel("Number of flows (log scale)")
ax.set_title("CICIDS2017 class distribution after cleaning")
for i, v in enumerate(d["clean_count"]):
    ax.text(v, i, f" {int(v):,}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_class_distribution.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
fam = df["attack_family"].value_counts()
ax.bar(fam.index.astype(str), fam.values)
ax.set_yscale("log")
ax.set_ylabel("Number of flows (log scale)")
ax.set_title("Flows per attack family")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_attack_families.png", dpi=200)
plt.show()

# Attack share per capture file: shows that attack types are concentrated on particular days,
# which matters for the train/test design in Notebook 02
per_day = pd.crosstab(df["source_file"], df["is_attack"]).rename(columns={0: "Benign", 1: "Attack"})
per_day.to_csv(f"{OUT_DIR}/01_attack_share_by_file.csv")
ax = per_day.plot(kind="barh", stacked=True, figsize=(10, 5))
ax.set_xlabel("Number of flows")
ax.set_title("Benign and attack flows per capture file")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_attack_share_by_file.png", dpi=200)
plt.show()

## 9. Feature overview
Feature ranges differ by many orders of magnitude, which is why scaling matters for distance-based and linear models but not for tree-based models. Highly correlated pairs are listed as evidence for the feature discussion; they are not dropped here because tree models are unaffected and the choice belongs in Notebook 02.

In [ ]:
desc = df[feature_cols].describe().T[["mean", "std", "min", "max"]]
desc.to_csv(f"{OUT_DIR}/01_feature_summary.csv")
display(desc.sort_values("max", ascending=False).head(15))

# Correlation on a random sample keeps this fast; 200k rows gives stable estimates
sample = df[feature_cols].sample(n=min(200_000, len(df)), random_state=RANDOM_STATE)
corr = sample.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
high_corr = (upper.stack().rename("abs_corr").reset_index()
             .rename(columns={"level_0": "feature_a", "level_1": "feature_b"}))
high_corr = high_corr[high_corr["abs_corr"] >= 0.99].sort_values("abs_corr", ascending=False)
high_corr.to_csv(f"{OUT_DIR}/01_high_correlation_pairs.csv", index=False)
print(f"Feature pairs with |r| >= 0.99: {len(high_corr)}")
display(high_corr.head(20))
del sample, corr, upper

## 10. Save the cleaned dataset for Notebook 02

In [ ]:
meta_cols = [c for c in [PORT_COL] if c in df.columns and c not in feature_cols]
keep = feature_cols + meta_cols + [LABEL_COL, "attack_family", "is_attack", "source_file"]
try:
    df[keep].to_parquet("data/cicids2017_clean.parquet", index=False)
    saved = "data/cicids2017_clean.parquet"
except ImportError:
    df[keep].to_pickle("data/cicids2017_clean.pkl")
    saved = "data/cicids2017_clean.pkl"
print("Saved", saved)

with open(f"{OUT_DIR}/01_feature_list.txt", "w") as fh:
    fh.write("\n".join(feature_cols))

# Record the settings used, so the report and Notebook 02 refer to the same configuration
with open(f"{OUT_DIR}/01_run_config.json", "w") as fh:
    json.dump({"files": file_summary["file"].tolist(), "rows_final": int(len(df)),
               "n_features": len(feature_cols), "metadata_columns": meta_cols,
               "exclude_destination_port": EXCLUDE_DESTINATION_PORT,
               "drop_conflicting_labels": DROP_CONFLICTING_LABELS,
               "pandas_version": pd.__version__}, fh, indent=2)
print(f"{len(feature_cols)} features, {len(df):,} rows")

## Notes for the report (fill in from your own outputs)
- Files used and row counts: `outputs/01_files_loaded.csv`
- Missing, infinite and constant values: `outputs/01_quality_issues.csv`
- Rows and columns removed at each step, and why: `outputs/01_cleaning_log.csv`
- Conflicting labels found: `outputs/01_conflicting_labels.csv` (link to Engelen, Rimmer and Joosen (2021) on CICIDS2017 labelling problems)
- Class imbalance before and after cleaning: `outputs/01_class_distribution.csv`
- Attacks concentrated on particular days: `outputs/01_attack_share_by_file.csv`
- Highly correlated features: `outputs/01_high_correlation_pairs.csv`
- Anything unexpected you noticed: